In [1]:
!pip install yt-dlp

In [4]:
!pip install hume==0.6.1

ERROR: Ignored the following yanked versions: 0.7.0
ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.8,<3.10; 0.1.1 Requires-Python >=3.8,<3.10; 0.1.2 Requires-Python >=3.8,<3.10; 0.1.3 Requires-Python >=3.8,<3.10; 0.1.4 Requires-Python >=3.8,<3.10
ERROR: Could not find a version that satisfies the requirement hume==0.6.1 (from versions: 0.1.5, 0.1.6, 0.1.7, 0.2.0, 0.3.0, 0.3.1, 0.3.2, 0.3.3, 0.3.4, 0.3.5, 0.3.6, 0.3.7, 0.4.0, 0.4.1, 0.4.2, 0.5.0rc1, 0.5.0rc2, 0.5.0rc3, 0.5.0rc4, 0.5.0, 0.5.1rc1, 0.5.1, 0.6.0rc1, 0.6.0rc2, 0.6.0rc3, 0.6.0, 0.7.0rc0, 0.7.0rc1, 0.7.0rc2, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.7.5, 0.7.6, 0.7.7, 0.7.8, 0.7.10, 0.7.11, 0.7.12, 0.7.13, 0.8.0, 0.8.1, 0.8.2, 0.8.3, 0.8.4, 0.8.5, 0.8.6, 0.9.0, 0.9.1, 0.10.1, 0.10.2, 0.11.0, 0.11.1, 0.11.2, 0.11.3, 0.11.4, 0.11.5, 0.11.6, 0.11.7, 0.12.0, 0.12.1, 0.13.0, 0.13.1, 0.13.2, 0.13.3, 0.13.4, 0.13.5, 0.13.6, 0.13.7, 0.13.8, 0.13.9, 0.13.10, 0.13.11, 0.13.12, 0.13.13, 0.14.

In [ ]:
import os
import asyncio
import json
import time
from hume import HumeBatchClient
from hume.models.config import FaceConfig
from google.colab import userdata

# Set your API key
API_KEY = userdata.get('HUME_API_KEY')

# Initialize the modern Batch Client
client = HumeBatchClient(API_KEY)

print("Latest SDK initialized successfully!")

In [ ]:
!pip install yt-dlp

import yt_dlp

def download_youtube_video(url: str, output_path: str = "video.mp4") -> str:
    ydl_opts = {
        'format': 'best[ext=mp4]',
        'outtmpl': output_path,
        'quiet': False,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    print(f"Downloaded: {output_path}")
    return output_path

VIDEO_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # Replace this
video_path = download_youtube_video(VIDEO_URL)

In [ ]:
def submit_job(video_path: str):
    config = FaceConfig()
    # In 0.14.0, we use submit_job directly
    job = client.submit_job(configs=[config], files=[video_path])
    print(f"Job submitted. Job ID: {job.job_id}")
    return job

job = submit_job(video_path)

In [ ]:
def wait_for_job(job):
    print("Waiting for job to complete...")
    while True:
        details = job.get_details()
        status = details.state.status
        print(f"Status: {status}")

        if status == "COMPLETED":
            print("Job completed!")
            break
        elif status == "FAILED":
            raise Exception("Job failed!")
        time.sleep(5)

wait_for_job(job)

In [ ]:
def get_and_display_results(job, top_n=5):
    predictions = job.get_predictions()

    # Save to file
    with open("results.json", "w") as f:
        json.dump(predictions, f, indent=2)
    print("Results saved to results.json\n")

    # Display top emotions per frame
    for file_result in predictions:
        for frame in file_result.get("results", {}).get("predictions", []):
            for face_pred in frame.get("models", {}).get("face", {}).get("grouped_predictions", []):
                for pred in face_pred.get("predictions", []):
                    timestamp = pred.get("frame", "N/A")
                    emotions = pred.get("emotions", [])
                    top_emotions = sorted(emotions, key=lambda e: e["score"], reverse=True)[:top_n]

                    print(f"Timestamp: {timestamp}s")
                    for e in top_emotions:
                        bar = "█" * int(e['score'] * 20)
                        print(f"  {e['name']:<20} {e['score']*100:.1f}%  {bar}")
                    print("-" * 40)

get_and_display_results(job)

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_emotions(predictions, emotions_to_track=["Joy", "Sadness", "Anger", "Fear", "Surprise"]):
    timeline = defaultdict(list)
    timestamps = []

    for file_result in predictions:
        for frame in file_result.get("results", {}).get("predictions", []):
            for face_pred in frame.get("models", {}).get("face", {}).get("grouped_predictions", []):
                for pred in face_pred.get("predictions", []):
                    t = pred.get("frame", 0)
                    timestamps.append(t)
                    emotion_map = {e["name"]: e["score"] for e in pred.get("emotions", [])}
                    for emo in emotions_to_track:
                        timeline[emo].append(emotion_map.get(emo, 0))

    plt.figure(figsize=(14, 6))
    for emo in emotions_to_track:
        plt.plot(timestamps, timeline[emo], label=emo)

    plt.xlabel("Time (seconds)")
    plt.ylabel("Emotion Score")
    plt.title("Emotion Timeline from Video")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Load saved results and plot
with open("results.json") as f:
    predictions = json.load(f)

plot_emotions(predictions)